In [38]:
import pickle
from pathlib import Path
import numpy as np
import numpy as np
import pandas as pd
import sklearn
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
import pickle
from pathlib import Path

import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
from torchvision import transforms, datasets

In [39]:
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

class CustomTensorDataset(Dataset):
    """TensorDataset with support of transforms.
    """
    def __init__(self, tensors, transform=None):
        # assert all(tensors[0].size(0) == tensor.size(0) for tensor in tensors)
        self.tensors = tensors
        self.transform = transform

    def __getitem__(self, index):
        x = self.tensors[0][index]

        if self.transform:
            x = self.transform(x)

        y = self.tensors[1][index]

        return x, y

    def __len__(self):
        return self.tensors[0].size(0)



def load_cifar_10_data(data_dir="/home/pt202342/data/cifar-10-batches-py", negatives=False):
    """
    Return train_data, train_filenames, train_labels, test_data, test_filenames, test_labels
    Source: https://github.com/snatch59/load-cifar-10/blob/master/load_cifar_10.py
    """

    # get the meta_data_dict
    # num_cases_per_batch: 1000
    # label_names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
    # num_vis: :3072

    meta_data_dict = unpickle(data_dir + "/batches.meta")
    cifar_label_names = meta_data_dict[b'label_names']
    cifar_label_names = np.array(cifar_label_names)

    # training data
    cifar_train_data = None
    cifar_train_filenames = []
    cifar_train_labels = []

    # cifar_train_data_dict
    # 'batch_label': 'training batch 5 of 5'
    # 'data': ndarray
    # 'filenames': list
    # 'labels': list

    for i in range(1, 6):
        cifar_train_data_dict = unpickle(data_dir + "/data_batch_{}".format(i))
        if i == 1:
            cifar_train_data = cifar_train_data_dict[b'data']
        else:
            cifar_train_data = np.vstack((cifar_train_data, cifar_train_data_dict[b'data']))
        cifar_train_filenames += cifar_train_data_dict[b'filenames']
        cifar_train_labels += cifar_train_data_dict[b'labels']

    cifar_train_data = cifar_train_data.reshape((len(cifar_train_data), 3, 32, 32))
    if negatives:
        cifar_train_data = cifar_train_data.transpose(0, 2, 3, 1).astype(np.float32)
    else:
        cifar_train_data = np.rollaxis(cifar_train_data, 1, 4)
    cifar_train_filenames = np.array(cifar_train_filenames)
    cifar_train_labels = np.array(cifar_train_labels)


    cifar_test_data_dict = unpickle(data_dir + "/test_batch")
    cifar_test_data = cifar_test_data_dict[b'data']
    cifar_test_filenames = cifar_test_data_dict[b'filenames']
    cifar_test_labels = cifar_test_data_dict[b'labels']

    cifar_test_data = cifar_test_data.reshape((len(cifar_test_data), 3, 32, 32))
    if negatives:
        cifar_test_data = cifar_test_data.transpose(0, 2, 3, 1).astype(np.float32)
    else:
        cifar_test_data = np.rollaxis(cifar_test_data, 1, 4)
    cifar_test_filenames = np.array(cifar_test_filenames)
    cifar_test_labels = np.array(cifar_test_labels)

    return cifar_train_data, cifar_train_filenames, cifar_train_labels, \
        cifar_test_data, cifar_test_filenames, cifar_test_labels, cifar_label_names

train_data, train_filenames, train_labels, test_data, test_filenames, test_labels, label_names = load_cifar_10_data()

In [40]:
train_data.shape

(50000, 32, 32, 3)

In [77]:
def get_loader(x, y):
    tensor_x = torch.Tensor(x)
    tensor_y = torch.Tensor(y)

    my_dataset = CustomTensorDataset(
        tensors=(tensor_x, tensor_y),
        transform=transforms.Compose(
            [
                # transforms.ToPILImage(),
                # transforms.ToTensor(),
                # transforms.Pad(2),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]),
        )
    
    loader = torch.utils.data.DataLoader(
        my_dataset, 
        shuffle=False,
        # batch_size=batch_size,
    )
    
    return loader

train_loader = get_loader(torch.Tensor(train_data).permute((0, 3, 1, 2)) / 255, train_labels)
test_loader = get_loader(torch.Tensor(test_data).permute((0, 3, 1, 2)) / 255, test_labels)
    



In [78]:
np.save("../data/cifar_train_data.npy", train_loader.dataset[:][0].numpy())
np.save("../data/cifar_train_labels.npy", train_loader.dataset[:][1].numpy())
np.save("../data/cifar_test_data.npy", test_loader.dataset[:][0].numpy())
np.save("../data/cifar_test_labels.npy", test_loader.dataset[:][1].numpy())

In [79]:
np.load("../data/cifar_test_data.npy").shape

(10000, 3, 36, 36)

In [81]:
np.load("/home/pt202342/projects/lidl/results/maf/aldi-gaussian_saw-1-3-02-11/data.npy").shape

(12000, 3)